<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 6 — Python Toolkit</div>
</div>

#### Standardising, PCA with the SVD and scikit-learn, k-means, and comparing clusters with labels.
***

## Why this notebook

Five pieces of Python for today's lab and for Exercise 2, Task 5. Fifteen minutes; skim what you know. Section 6 has optional practice with numbers to check against. If arrays, slicing, loops or functions themselves feel shaky, start with the matching section of the [Python foundations notebook](https://colab.research.google.com/github/MatSmithAstro/phys7101_2026_resources/blob/main/workshops/python_foundations/python_foundations.ipynb) first.

1. standardising: by hand, and with scikit-learn;
2. `np.linalg.svd`: principal components in three lines;
3. scikit-learn's `PCA`: the same, with names;
4. scikit-learn's `KMeans`;
5. comparing clusters with known labels: `pd.crosstab`, and why cluster numbers are arbitrary.

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

rng = np.random.default_rng(0)
# a small table: 200 objects, three quantities in very different units
X = np.column_stack([rng.normal(20, 2, 200),            # e.g. a temperature (degrees)
                     rng.normal(1000, 300, 200),        # e.g. a pressure (Pa)
                     rng.normal(0.5, 0.1, 200)])        # e.g. a fraction
print(X[:3].round(2))

## <span style="color:#B5121B">1. Standardising</span>

Subtract each column's mean and divide by its standard deviation: every column then has mean 0 and standard deviation 1. `axis=0` means "down each column". scikit-learn's `StandardScaler` does the same, but divides by the standard deviation with $N$ rather than $N-1$ (`ddof=0`): for hundreds of rows the difference is negligible.

In [ ]:
Z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
print("means:", Z.mean(axis=0).round(10), " standard deviations:", Z.std(axis=0, ddof=1).round(10))
Z_sk = StandardScaler().fit_transform(X)
print("largest difference from StandardScaler:", np.abs(Z - Z_sk).max().round(4))

## <span style="color:#B5121B">2. Principal components with the SVD</span>

For a standardised table `Z` (rows are objects, columns are quantities), `np.linalg.svd` returns `U, s, Vt`. Each **row** of `Vt` is a principal direction; its entries are the **loadings** (the weight of each quantity). The variance along direction $k$ is $s_k^2/(N-1)$. The coordinates of each object on the components (its *scores*) are `Z @ Vt.T`.

In [ ]:
U, s, Vt = np.linalg.svd(Z, full_matrices=False)
var = s**2 / (len(Z) - 1)
print("variance along each direction:", var.round(3), " fraction:", (var / var.sum()).round(3))
print(pd.DataFrame(Vt.T, index=["temperature", "pressure", "fraction"], columns=["PC1", "PC2", "PC3"]).round(2))
scores = Z @ Vt.T
print("scores of the first object:", scores[0].round(3))

In [ ]:
# Your turn: these three columns are independent, so what do you expect the three variances to be? Why aren't they exactly that?

## <span style="color:#B5121B">3. scikit-learn's PCA</span>

`PCA().fit(Z)` gives the same answer with names: `explained_variance_ratio_` (fractions), `components_` (the rows of `Vt`: loadings) and `transform(Z)` (the scores). **The sign of each component is arbitrary**: a direction and its opposite describe the same axis, so don't be surprised if your loadings are all flipped.

In [ ]:
pca = PCA().fit(Z)
print("fractions:", pca.explained_variance_ratio_.round(3))
print("same directions as the SVD (up to sign):", np.allclose(np.abs(pca.components_), np.abs(Vt)))

## <span style="color:#B5121B">4. k-means</span>

`KMeans(n_clusters=k, n_init=10, random_state=0)` runs k-means from 10 starting points and keeps the best. `fit_predict` returns a cluster number for each row; `cluster_centers_` holds the centres and `inertia_` the total squared distance of points from their centres (smaller is tighter). Fix `random_state` so that your results are reproducible.

In [ ]:
# three groups in two dimensions, so we know the answer
truth = np.repeat([0, 1, 2], 100)
centres = np.array([[0, 0], [4, 0], [2, 3]])
P = centres[truth] + rng.normal(0, 0.7, (300, 2))
km = KMeans(n_clusters=3, n_init=10, random_state=0)
labels = km.fit_predict(P)
print("centres found:", km.cluster_centers_.round(2).tolist(), " inertia:", round(km.inertia_, 1))
plt.figure(figsize=(4.5, 3.5)); plt.scatter(P[:, 0], P[:, 1], c=labels, s=8); plt.show()

## <span style="color:#B5121B">5. Comparing clusters with labels</span>

`pd.crosstab(truth, labels)` counts how many objects of each true class fall in each cluster. **Cluster numbers are arbitrary**: k-means may call your class 0 "cluster 2". To count matches, try every way of naming the clusters and keep the best (`itertools.permutations`), or name each cluster after the largest class in it.

In [ ]:
print(pd.crosstab(truth, labels, rownames=["true class"], colnames=["cluster"]))
best = max(sum(np.sum((labels == p[i]) & (truth == i)) for i in range(3)) for p in itertools.permutations(range(3)))
print("matched:", best, "of", len(truth))

## <span style="color:#FD9029">6. Optional practice</span>

Each has a number to check against.

**P1.** Draw 1000 pairs $(x, y)$ from a Gaussian with standard deviations 1 and correlation 0.9 (`rng.multivariate_normal([0, 0], [[1, 0.9], [0.9, 1]], 1000)`). Do a PCA. What are the two variances, and what is the first direction?

In [ ]:
# P1

**Check:** variances about 1.9 and 0.1 (that is, $1 \pm 0.9$); the first direction is about $(0.71, 0.71)$ (or its opposite): the diagonal along which $x$ and $y$ vary together.

**P2.** Multiply the $y$ values from P1 by 1000 (a change of units) and repeat *without* standardising. What does the first direction become? Then standardise and repeat.

In [ ]:
# P2

**Check:** raw: almost exactly the $y$ axis, $(0.001, 1.000)$: the units decide. Standardised: back to $(0.71, 0.71)$.

**P3.** Cluster the points `P` from Section 4 with $k = 2, 3, 4, 5$ and print the inertia of each. By how much does it fall from $k = 2$ to $3$, and from $3$ to $4$?

In [ ]:
# P3

**Check:** the inertia falls by a factor of about 3.5 from $k = 2$ to $3$, but only by about 1.2 from $3$ to $4$: here there is a clear 'elbow' at $k = 3$, because the three groups really are round and separate. Real data are rarely this tidy.

**P4.** Write a function `named_after_largest(truth, labels)` that names each cluster after the largest true class in it and returns how many objects are then in a cluster named after their own class. Check it on `truth` and `labels` from Section 5.

In [ ]:
# P4

**Check:** the same number as the best match in Section 5 when every cluster is mostly one class; it can be larger when two clusters share a class (as with the two galaxy clusters in the lab).

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>